In [1]:
print(1)

1


In [14]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import to_date,col

In [7]:
data = [

    (1, "C001", "2026-01-05", 100),

    (2, "C002", "2026-01-10", 200),

    (3, "C003", "2026-01-15", 150),

    (4, "C004", "2026-01-20", 300),

    (5, "C001", "2026-02-03", 120),

    (6, "C003", "2026-02-12", 180),

    (7, "C005", "2026-02-18", 220),

    (8, "C006", "2026-03-01", 500)

]

columns = ["order_id", "customer_id", "order_date", "amount"]



In [9]:
data,columns

([(1, 'C001', '2026-01-05', 100),
  (2, 'C002', '2026-01-10', 200),
  (3, 'C003', '2026-01-15', 150),
  (4, 'C004', '2026-01-20', 300),
  (5, 'C001', '2026-02-03', 120),
  (6, 'C003', '2026-02-12', 180),
  (7, 'C005', '2026-02-18', 220),
  (8, 'C006', '2026-03-01', 500)],
 ['order_id', 'customer_id', 'order_date', 'amount'])

In [12]:
spark = SparkSession.builder.appName("JanNotFebExample").getOrCreate()

26/06/12 23:03:41 WARN Utils: Your hostname, MKHOMBP14018.local resolves to a loopback address: 127.0.0.1; using 192.168.1.38 instead (on interface en0)
26/06/12 23:03:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/12 23:03:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [17]:
columns = ["order_id", "customer_id", "order_date", "amount"]

orders_df = spark.createDataFrame(data, columns)

orders_df = orders_df.withColumn("order_date", to_date(col("order_date")))

orders_df.show()

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|       1|       C001|2026-01-05|   100|
|       2|       C002|2026-01-10|   200|
|       3|       C003|2026-01-15|   150|
|       4|       C004|2026-01-20|   300|
|       5|       C001|2026-02-03|   120|
|       6|       C003|2026-02-12|   180|
|       7|       C005|2026-02-18|   220|
|       8|       C006|2026-03-01|   500|
+--------+-----------+----------+------+



In [19]:
orders_df = orders_df.withColumn("order_date", to_date("order_date"))

orders_df.createOrReplaceTempView("orders")

display(orders_df)

DataFrame[order_id: bigint, customer_id: string, order_date: date, amount: bigint]

In [21]:
spark.sql("select * from orders" ).show()

+--------+-----------+----------+------+
|order_id|customer_id|order_date|amount|
+--------+-----------+----------+------+
|       1|       C001|2026-01-05|   100|
|       2|       C002|2026-01-10|   200|
|       3|       C003|2026-01-15|   150|
|       4|       C004|2026-01-20|   300|
|       5|       C001|2026-02-03|   120|
|       6|       C003|2026-02-12|   180|
|       7|       C005|2026-02-18|   220|
|       8|       C006|2026-03-01|   500|
+--------+-----------+----------+------+



In [25]:
query="""
with  jan_data   as 
(
select customer_id From orders where order_date between '2026-01-01' and '2026-01-31' group by customer_id
),

feb_data   as 
(
select customer_id From orders where order_date between '2026-02-01' and '2026-02-28' group by customer_id
)
Select a.customer_id
FROM jan_data as a
LEFT JOIN feb_data as b 
ON (a.customer_id=b.customer_id)
Where b.customer_id IS NULL

"""

In [26]:
spark.sql(query ).show()

+-----------+
|customer_id|
+-----------+
|       C002|
|       C004|
+-----------+



In [27]:
from pyspark.sql.functions import col

jan_customers = (
    orders_df
    .filter((col("order_date") >= "2026-01-01") & (col("order_date") < "2026-02-01"))
    .select("customer_id")
    .distinct()
)

feb_customers = (
    orders_df
    .filter((col("order_date") >= "2026-02-01") & (col("order_date") < "2026-03-01"))
    .select("customer_id")
    .distinct()
)

result = jan_customers.join(
    feb_customers,
    on="customer_id",
    how="left_anti"
)

display(result)

DataFrame[customer_id: string]

In [29]:
result.show()

+-----------+
|customer_id|
+-----------+
|       C002|
|       C004|
+-----------+



In [34]:
from pyspark.sql.functions import col

data = [

    (1, "Alice", 90000, None),   # CEO, no manager

    (2, "Bob", 70000, 1),       # Bob reports to Alice

    (3, "Charlie", 95000, 1),   # Charlie reports to Alice

    (4, "David", 60000, 2),     # David reports to Bob

    (5, "Eva", 80000, 2),       # Eva reports to Bob

    (6, "Frank", 50000, 3)      # Frank reports to Charlie

]

columns = ["emp_id", "emp_name", "salary", "manager_id"]

employees_df = spark.createDataFrame(data, columns)
employees_df.createOrReplaceTempView("employee")

employees_df.show()

+------+--------+------+----------+
|emp_id|emp_name|salary|manager_id|
+------+--------+------+----------+
|     1|   Alice| 90000|      NULL|
|     2|     Bob| 70000|         1|
|     3| Charlie| 95000|         1|
|     4|   David| 60000|         2|
|     5|     Eva| 80000|         2|
|     6|   Frank| 50000|         3|
+------+--------+------+----------+



In [61]:
query2="""

select e.emp_id, e.emp_name,e.salary ,m.salary from employee as e
LEFT JOIN employee as m  on e.manager_id=m.emp_id
where e.salary > m.salary
 
"""

In [62]:
spark.sql(query2).show()

+------+--------+------+------+
|emp_id|emp_name|salary|salary|
+------+--------+------+------+
|     3| Charlie| 95000| 90000|
|     5|     Eva| 80000| 70000|
+------+--------+------+------+



In [63]:
#Given users and transactions tables, find users who have never made a transaction.


In [64]:
users_data = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David"),
    (5, "Eva")
]

users_columns = ["user_id", "user_name"]

users_df = spark.createDataFrame(users_data, users_columns)

users_df.show()

+-------+---------+
|user_id|user_name|
+-------+---------+
|      1|    Alice|
|      2|      Bob|
|      3|  Charlie|
|      4|    David|
|      5|      Eva|
+-------+---------+



In [65]:
transactions_data = [

    (101, 1, 500),

    (102, 1, 300),

    (103, 3, 700),

    (104, 5, 200)

]

transactions_columns = ["transaction_id", "user_id", "amount"]

transactions_df = spark.createDataFrame(transactions_data, transactions_columns)

transactions_df.show()

+--------------+-------+------+
|transaction_id|user_id|amount|
+--------------+-------+------+
|           101|      1|   500|
|           102|      1|   300|
|           103|      3|   700|
|           104|      5|   200|
+--------------+-------+------+



In [66]:
users_df.createOrReplaceTempView("users")
transactions_df.createOrReplaceTempView("transactions")

In [75]:
query3 = """ 
select * from users as u
LEFT JOIN transactions as t
ON t.user_id=u.user_id
where t.user_id IS NULL


"""

In [76]:
spark.sql(query3).show()

+-------+---------+--------------+-------+------+
|user_id|user_name|transaction_id|user_id|amount|
+-------+---------+--------------+-------+------+
|      2|      Bob|          NULL|   NULL|  NULL|
|      4|    David|          NULL|   NULL|  NULL|
+-------+---------+--------------+-------+------+



In [77]:
result = users_df.join(
    transactions_df.select("user_id").distinct(),
    on="user_id",
    how="left_anti"
)

result.show()

+-------+---------+
|user_id|user_name|
+-------+---------+
|      2|      Bob|
|      4|    David|
+-------+---------+



In [78]:
users_data = [
    (1, "Alice"),
    (2, "Bob"),
    (3, "Charlie"),
    (4, "David")
]

users_df = spark.createDataFrame(users_data, ["user_id", "user_name"])

transactions_data = [
    (101, 1),
    (102, 3),
    (103, None)
]

transactions_df = spark.createDataFrame(transactions_data, ["transaction_id", "user_id"])

users_df.createOrReplaceTempView("users")
transactions_df.createOrReplaceTempView("transactions")

print("Users")
users_df.show()

print("Transactions")
transactions_df.show()

Users
+-------+---------+
|user_id|user_name|
+-------+---------+
|      1|    Alice|
|      2|      Bob|
|      3|  Charlie|
|      4|    David|
+-------+---------+

Transactions
+--------------+-------+
|transaction_id|user_id|
+--------------+-------+
|           101|      1|
|           102|      3|
|           103|   NULL|
+--------------+-------+



In [82]:
query5="""SELECT *
FROM users
WHERE user_id NOT IN (
    SELECT user_id
    FROM transactions
) """

In [84]:
spark.sql(query5).show()

+-------+---------+
|user_id|user_name|
+-------+---------+
+-------+---------+



In [85]:
query5="""SELECT *
FROM users
WHERE user_id NOT IN (
    SELECT user_id
    FROM transactions where user_id IS NOT NULL 
) """
spark.sql(query5).show()

+-------+---------+
|user_id|user_name|
+-------+---------+
|      2|      Bob|
|      4|    David|
+-------+---------+



In [88]:
query5="""
 SELECT
    u.user_id,
    u.user_name
FROM users u
WHERE NOT EXISTS (
    SELECT 1
    FROM transactions t
    WHERE t.user_id = u.user_id
) """
spark.sql(query5).show()

+-------+---------+
|user_id|user_name|
+-------+---------+
|      2|      Bob|
|      4|    David|
+-------+---------+



In [86]:
users_data = [

    (1, "Alice"),

    (2, "Bob"),

    (3, "Charlie"),

    (4, "David")

]

users_df = spark.createDataFrame(users_data, ["user_id", "user_name"])

transactions_data = [

    (101, 1),

    (102, 3),

    (103, None)

]

transactions_df = spark.createDataFrame(transactions_data, ["transaction_id", "user_id"])

print("Users")

users_df.show()

print("Transactions")

transactions_df.show()

# Wrong-looking result with NOT IN

users_df.createOrReplaceTempView("users")

transactions_df.createOrReplaceTempView("transactions")

print("Correct result using PySpark left anti join")

result = users_df.join(

    transactions_df

        .select("user_id")

        .where("user_id IS NOT NULL")

        .distinct(),

    on="user_id",

    how="left_anti"

)

result.show()

Users
+-------+---------+
|user_id|user_name|
+-------+---------+
|      1|    Alice|
|      2|      Bob|
|      3|  Charlie|
|      4|    David|
+-------+---------+

Transactions
+--------------+-------+
|transaction_id|user_id|
+--------------+-------+
|           101|      1|
|           102|      3|
|           103|   NULL|
+--------------+-------+

Correct result using PySpark left anti join
+-------+---------+
|user_id|user_name|
+-------+---------+
|      2|      Bob|
|      4|    David|
+-------+---------+



In [89]:
from pyspark.sql.functions import col, to_date

transactions_data = [
    (1, "P001", "2026-01-05", 2),
    (2, "P001", "2026-01-20", 1),
    (3, "P001", "2026-02-10", 3),
    (4, "P002", "2026-01-15", 5),
    (5, "P002", "2026-02-05", 2)
]

transactions_columns = ["txn_id", "product_id", "txn_date", "quantity"]

transactions_df = spark.createDataFrame(transactions_data, transactions_columns)

transactions_df = transactions_df.withColumn(
    "txn_date",
    to_date(col("txn_date"))
)

price_data = [
    ("P001", "2026-01-01", "2026-01-31", 100),
    ("P001", "2026-02-01", "2026-02-28", 120),
    ("P002", "2026-01-01", "2026-01-31", 50),
    ("P002", "2026-02-01", "2026-02-28", 60)
]

price_columns = ["product_id", "start_date", "end_date", "price"]

price_df = spark.createDataFrame(price_data, price_columns)

price_df = (
    price_df
    .withColumn("start_date", to_date(col("start_date")))
    .withColumn("end_date", to_date(col("end_date")))
)

print("Transactions")
transactions_df.show()

print("Price History")
price_df.show()

Transactions
+------+----------+----------+--------+
|txn_id|product_id|  txn_date|quantity|
+------+----------+----------+--------+
|     1|      P001|2026-01-05|       2|
|     2|      P001|2026-01-20|       1|
|     3|      P001|2026-02-10|       3|
|     4|      P002|2026-01-15|       5|
|     5|      P002|2026-02-05|       2|
+------+----------+----------+--------+

Price History
+----------+----------+----------+-----+
|product_id|start_date|  end_date|price|
+----------+----------+----------+-----+
|      P001|2026-01-01|2026-01-31|  100|
|      P001|2026-02-01|2026-02-28|  120|
|      P002|2026-01-01|2026-01-31|   50|
|      P002|2026-02-01|2026-02-28|   60|
+----------+----------+----------+-----+



In [91]:
result = (
    transactions_df.alias("t")
    .join(
        price_df.alias("p"),
        (col("t.product_id") == col("p.product_id")) &
        (col("t.txn_date") >= col("p.start_date")) &
        (col("t.txn_date") <= col("p.end_date")),
        "left"
    )
    .select(
        col("t.txn_id"),
        col("t.product_id"),
        col("t.txn_date"),
        col("t.quantity"),
        col("p.price"),
        (col("t.quantity") * col("p.price")).alias("total_amount")
    )
)

result.show()

+------+----------+----------+--------+-----+------------+
|txn_id|product_id|  txn_date|quantity|price|total_amount|
+------+----------+----------+--------+-----+------------+
|     1|      P001|2026-01-05|       2|  100|         200|
|     2|      P001|2026-01-20|       1|  100|         100|
|     3|      P001|2026-02-10|       3|  120|         360|
|     4|      P002|2026-01-15|       5|   50|         250|
|     5|      P002|2026-02-05|       2|   60|         120|
+------+----------+----------+--------+-----+------------+



In [92]:
transactions_df.createOrReplaceTempView("transactions")
price_df.createOrReplaceTempView("price_history")

In [96]:
sql6="""
SELECT
    t.txn_id,
    t.product_id,
    t.txn_date,
    t.quantity,
    p.price,
    t.quantity * p.price AS total_amount
FROM transactions t
LEFT JOIN price_history p
    ON t.product_id = p.product_id
   AND t.txn_date >= p.start_date
   AND t.txn_date <= p.end_date
"""
spark.sql(sql6).show()

+------+----------+----------+--------+-----+------------+
|txn_id|product_id|  txn_date|quantity|price|total_amount|
+------+----------+----------+--------+-----+------------+
|     1|      P001|2026-01-05|       2|  100|         200|
|     2|      P001|2026-01-20|       1|  100|         100|
|     3|      P001|2026-02-10|       3|  120|         360|
|     4|      P002|2026-01-15|       5|   50|         250|
|     5|      P002|2026-02-05|       2|   60|         120|
+------+----------+----------+--------+-----+------------+

